# Week 6 Exercise — The Price Is Right: Product Price Predictor

**By:** Jaymineh  
**API:** OpenAI (fine-tuning GPT-4o-mini) + OpenRouter (zero-shot pricing)

A product price predictor that compares traditional ML, zero-shot LLM pricing, and
fine-tuned model approaches using Amazon product data.

**Week 6 skills demonstrated:**
- Loading curated product data from HuggingFace Hub
- Traditional ML baseline with **XGBoost** (feature engineering + NLP)
- **Zero-shot LLM pricing** via OpenRouter
- **Fine-tuning GPT-4o-mini** via OpenAI API with JSONL training data
- Evaluation using MAE, scatter plots, and the `pricer.evaluator` framework
- Comparing all approaches side by side

## Step 1: Setup & Load Data

In [1]:
import sys
from pathlib import Path

# Add week6 root to path so pricer module is importable
week6_dir = str(Path.cwd().parent.parent)
if week6_dir not in sys.path:
    sys.path.insert(0, week6_dir)
print(f"Added to path: {week6_dir}")

Added to path: c:\Users\BOSS\Desktop\DevOps\llm_engineering\week6


In [2]:
import os
import json
import re
import time
import random
import numpy as np
from dotenv import load_dotenv
from openai import OpenAI
from litellm import completion
import plotly.graph_objects as go

load_dotenv(override=True)

openai_key = os.getenv('OPENAI_API_KEY')
openrouter_key = os.getenv('OPENROUTER_API_KEY')
print(f"OpenAI key: {'YES' if openai_key else 'NO'}")
print(f"OpenRouter key: {'YES' if openrouter_key else 'NO'}")

OpenAI key: YES
OpenRouter key: YES


In [3]:
from pricer.items import Item
from pricer.evaluator import Tester, evaluate

train, val, test = Item.from_hub('ed-donner/items_lite')
print(f"Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")
print(f"\nSample item: {train[0].title}")
print(f"Price: ${train[0].price}")
print(f"Summary: {train[0].summary[:200]}...")

Train: 20,000 | Val: 1,000 | Test: 1,000

Sample item: Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)
Price: $64.3
Summary: Title: Schlage F59 & 613 Andover Interior Knob (Deadbolt Included)  
Category: Home Hardware  
Brand: Schlage  
Description: A single‑piece oil‑rubbed bronze knob that mounts to a deadbolt for secure,...


In [4]:
# Quick EDA: price distribution
prices = [item.price for item in train]
print(f"Price range: ${min(prices):.2f} - ${max(prices):.2f}")
print(f"Mean: ${np.mean(prices):.2f}")
print(f"Median: ${np.median(prices):.2f}")
print(f"Std: ${np.std(prices):.2f}")

Price range: $0.73 - $999.00
Mean: $140.35
Median: $79.99
Std: $160.84


## Step 2: Baselines & Traditional ML (Day 3 Concepts)

In [5]:
# Baseline 1: Always predict the mean price
average_price = np.mean([item.price for item in train])

def constant_pricer(item):
    return average_price

print(f"Constant prediction: ${average_price:.2f}")
evaluate(constant_pricer, test, size=200)

Constant prediction: $140.35


  0%|          | 0/200 [00:00<?, ?it/s]

$79 $24 $85 $70 $110 $90 $4 $75 $105 $190 $573 $239 $120 $86 $61 $107 $61 $90 $70 $21 $6 $16 $55 $35 $192 $312 $355 $120 $42 $60 $120 $80 $20 $60 $25 $679 $81 $84 $73 $102 $59 $60 $105 $115 $80 $115 $122 $109 $5 $62 $105 $10 $335 $21 $87 $6 $133 $100 $62 $128 $96 $62 $49 $30 $488 $50 $100 $305 $15 $64 $109 $123 $140 $121 $90 $104 $16 $130 $123 $122 $20 $128 $110 $41 $113 $80 $42 $166 $20 $95 $118 $45 $120 $105 $132 $88 $106 $17 $130 $435 $40 $23 $103 $1 $109 $22 $115 $260 $109 $159 $80 $174 $109 $12 $55 $30 $116 $120 $84 $37 $124 $51 $70 $26 $60 $80 $120 $41 $39 $1 $69 $3 $55 $110 $75 $125 $65 $70 $12 $2 $76 $110 $60 $121 $54 $108 $95 $370 $125 $107 $121 $34 $93 $0 $121 $139 $91 $84 $179 $90 $149 $114 $98 $127 $700 $117 $307 $90 $100 $130 $115 $118 $280 $118 $39 $9 $112 $47 $46 $47 $514 $115 $160 $108 $90 $118 $7 $73 $80 $114 $105 $89 $105 $2 $40 $60 $30 $140 $89 $114 

In [6]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_extraction.text import CountVectorizer

# Build text features from summaries
train_texts = [item.summary or item.title for item in train]
train_prices = np.array([item.price for item in train])

vectorizer = CountVectorizer(max_features=2000, stop_words='english')
X_train = vectorizer.fit_transform(train_texts)

# Train Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, train_prices)
print(f"Random Forest trained on {X_train.shape[0]} items with {X_train.shape[1]} features")

def random_forest_pricer(item):
    text = item.summary or item.title
    X = vectorizer.transform([text])
    return rf.predict(X)[0]

evaluate(random_forest_pricer, test, size=200)

Random Forest trained on 20000 items with 2000 features


  0%|          | 0/200 [00:00<?, ?it/s]

$48 $81 $3 $9 $87 $150 $62 $75 $46 $262 $499 $281 $71 $69 $25 $5 $25 $74 $40 $14 $3 $13 $22 $6 $178 $328 $128 $48 $52 $47 $36 $73 $30 $0 $21 $305 $38 $52 $111 $54 $125 $65 $17 $65 $136 $35 $46 $25 $67 $28 $7 $10 $244 $24 $150 $70 $35 $159 $18 $55 $120 $18 $61 $29 $399 $43 $6 $116 $16 $118 $25 $19 $106 $134 $1 $186 $26 $13 $30 $80 $52 $75 $30 $52 $14 $16 $9 $124 $11 $142 $39 $87 $1 $42 $33 $88 $11 $17 $165 $302 $36 $0 $6 $77 $26 $8 $73 $216 $16 $170 $34 $25 $78 $34 $35 $22 $70 $15 $205 $8 $36 $24 $73 $44 $67 $21 $23 $39 $59 $13 $33 $94 $106 $25 $32 $90 $2 $11 $35 $52 $5 $176 $14 $96 $74 $64 $69 $343 $52 $10 $18 $98 $52 $86 $41 $76 $92 $20 $40 $3 $84 $1 $6 $25 $395 $31 $242 $30 $17 $30 $42 $8 $221 $25 $13 $48 $55 $8 $28 $7 $388 $13 $7 $30 $123 $64 $63 $96 $18 $37 $6 $73 $38 $3 $8 $26 $49 $36 $19 $28 

In [7]:
from xgboost import XGBRegressor

xgb = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    n_jobs=-1,
)
xgb.fit(X_train, train_prices)
print(f"XGBoost trained")

def xgboost_pricer(item):
    text = item.summary or item.title
    X = vectorizer.transform([text])
    return xgb.predict(X)[0]

evaluate(xgboost_pricer, test, size=200)

XGBoost trained


  0%|          | 0/200 [00:00<?, ?it/s]

$97 $43 $7 $5 $130 $144 $45 $28 $30 $37 $586 $238 $79 $117 $11 $20 $59 $21 $15 $48 $1 $53 $23 $77 $240 $304 $81 $41 $36 $24 $81 $82 $30 $25 $57 $167 $66 $97 $147 $95 $135 $87 $67 $26 $123 $60 $54 $63 $36 $14 $40 $38 $192 $7 $226 $77 $82 $188 $13 $40 $74 $32 $36 $29 $422 $54 $56 $229 $58 $117 $35 $66 $41 $127 $21 $142 $212 $34 $32 $73 $64 $102 $71 $36 $27 $35 $4 $141 $37 $216 $53 $87 $33 $24 $36 $88 $40 $11 $142 $223 $59 $50 $12 $42 $1 $66 $94 $205 $3 $103 $23 $70 $82 $61 $92 $51 $145 $43 $146 $57 $85 $85 $40 $19 $96 $34 $27 $86 $60 $42 $73 $41 $94 $63 $72 $78 $104 $40 $77 $49 $7 $128 $52 $98 $89 $86 $65 $336 $56 $24 $0 $126 $36 $50 $43 $135 $139 $31 $31 $37 $115 $8 $2 $16 $414 $21 $233 $63 $36 $35 $27 $36 $222 $24 $72 $50 $90 $6 $30 $55 $377 $32 $1 $43 $101 $91 $55 $39 $55 $53 $45 $76 $36 $13 $14 $37 $45 $15 $77 $53 

## Step 3: Zero-Shot LLM Pricing (Day 4 Concepts)

Ask an LLM to estimate the price from the product description alone, with no training.

In [8]:
def zero_shot_gpt4o_mini(item):
    text = item.summary or item.title
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{text}"
    try:
        response = completion(
            model="openrouter/openai/gpt-4o-mini",
            messages=[{"role": "user", "content": message}],
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"Error: {e}")
        return str(average_price)

# Test on one item first
sample = test[0]
result = zero_shot_gpt4o_mini(sample)
print(f"Product: {sample.title[:60]}...")
print(f"Actual: ${sample.price:.2f}")
print(f"Predicted: {result}")

Product: Old Blood Noise Excess V2 Distortion Chorus/Delay Pedal...
Actual: $219.00
Predicted: $199.99


In [9]:
# Run on 200 test items (takes ~2-3 min with threading)
evaluate(zero_shot_gpt4o_mini, test, size=200)

  0%|          | 0/200 [00:00<?, ?it/s]

$19 $34 $19 $10 $70 $110 $64 $65 $0 $20 $214 $79 $10 $29 $29 $3 $51 $0 $90 $31 $54 $6 $35 $25 $82 $253 $5 $6 $151 $64 $30 $20 $140 $54 $35 $219 $36 $30 $14 $22 $170 $54 $10 $5 $70 $5 $28 $4 $65 $52 $20 $100 $125 $20 $67 $16 $6 $50 $52 $14 $116 $12 $41 $40 $279 $5 $40 $295 $25 $74 $12 $1 $20 $7 $20 $10 $126 $0 $3 $4 $20 $4 $4 $79 $1 $10 $32 $56 $0 $1 $4 $45 $6 $15 $1 $78 $12 $97 $180 $325 $20 $47 $7 $49 $49 $82 $9 $325 $1 $99 $10 $136 $49 $22 $4 $130 $1 $4 $64 $28 $14 $211 $50 $81 $50 $15 $11 $51 $29 $89 $59 $13 $35 $10 $85 $11 $85 $30 $2 $162 $22 $0 $10 $4 $104 $48 $15 $10 $35 $8 $2 $144 $21 $10 $3 $29 $27 $30 $70 $5 $90 $17 $12 $3 $140 $2 $152 $20 $6 $10 $1 $3 $120 $13 $31 $51 $12 $18 $6 $43 $46 $25 $225 $49 $30 $18 $43 $13 $10 $11 $5 $44 $5 $61 $25 $70 $20 $30 $11 $4 

## Step 4: Fine-Tune GPT-4o-mini (Day 5 Concepts)

Create JSONL training data, upload to OpenAI, fine-tune, and evaluate.

In [10]:
# Create JSONL training and validation data
TRAIN_SIZE = 400
VAL_SIZE = 100

def make_jsonl_entry(item):
    text = item.summary or item.title
    return {
        "messages": [
            {"role": "user", "content": f"Estimate the price of this product. Respond with the price, no explanation\n\n{text}"},
            {"role": "assistant", "content": f"${item.price:.2f}"}
        ]
    }

# Sample training and validation data
random.seed(42)
train_sample = random.sample(train, TRAIN_SIZE)
val_sample = random.sample(val, VAL_SIZE)

train_jsonl_path = "fine_tune_train.jsonl"
val_jsonl_path = "fine_tune_val.jsonl"

for path, data in [(train_jsonl_path, train_sample), (val_jsonl_path, val_sample)]:
    with open(path, "w") as f:
        for item in data:
            f.write(json.dumps(make_jsonl_entry(item)) + "\n")

print(f"Created {train_jsonl_path} with {TRAIN_SIZE} examples")
print(f"Created {val_jsonl_path} with {VAL_SIZE} examples")

# Preview one entry
sample_entry = make_jsonl_entry(train_sample[0])
print(f"\nSample entry:")
print(json.dumps(sample_entry, indent=2)[:500])

Created fine_tune_train.jsonl with 400 examples
Created fine_tune_val.jsonl with 100 examples

Sample entry:
{
  "messages": [
    {
      "role": "user",
      "content": "Estimate the price of this product. Respond with the price, no explanation\n\nTitle: 25 Pack Small Cardboard Shipping Boxes 9\"L x 6\"W x 4\"H  \nCategory: Packaging & Shipping  \nBrand: KAKA SENLIN  \nDescription: Reusable, eco\u2011friendly corrugated mailing boxes with free memo stickers, ideal for shipping, storage, and gift wrapping.  \nDetails: Each box features 200\u202flb breaking strength, 32\u202fECT grade, rounded edges f


In [11]:
# Upload files to OpenAI
openai_client = OpenAI()

with open(train_jsonl_path, "rb") as f:
    train_file = openai_client.files.create(file=f, purpose="fine-tune")
print(f"Train file uploaded: {train_file.id}")

with open(val_jsonl_path, "rb") as f:
    val_file = openai_client.files.create(file=f, purpose="fine-tune")
print(f"Val file uploaded: {val_file.id}")

Train file uploaded: file-EPXi8wE1WeV4574judFDsU
Val file uploaded: file-FpqtFEevE6XBpew18S9puX


In [12]:
# Start fine-tuning job
ft_job = openai_client.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=val_file.id,
    model="gpt-4o-mini-2024-07-18",
    hyperparameters={
        "n_epochs": 1,
    },
)
print(f"Fine-tuning job created: {ft_job.id}")
print(f"Status: {ft_job.status}")

Fine-tuning job created: ftjob-NrbutElksgsaihNOH8vxPAGl
Status: validating_files


In [13]:
# Poll for completion (fine-tuning typically takes 5-15 min)
while True:
    job = openai_client.fine_tuning.jobs.retrieve(ft_job.id)
    print(f"Status: {job.status}")
    if job.status in ["succeeded", "failed", "cancelled"]:
        break
    time.sleep(30)

if job.status == "succeeded":
    fine_tuned_model = job.fine_tuned_model
    print(f"\nFine-tuned model: {fine_tuned_model}")
else:
    print(f"\nJob ended with status: {job.status}")
    if job.error:
        print(f"Error: {job.error}")

Status: validating_files
Status: validating_files
Status: validating_files
Status: validating_files
Status: validating_files
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: running
Status: succeeded

Fine-tuned model: ft:gpt-4o-mini-2024-07-18:personal::DG

In [14]:
# Evaluate the fine-tuned model
def fine_tuned_pricer(item):
    text = item.summary or item.title
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{text}"
    try:
        response = openai_client.chat.completions.create(
            model=fine_tuned_model,
            messages=[{"role": "user", "content": message}],
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"Error: {e}")
        return str(average_price)

# Test on one item
sample = test[0]
result = fine_tuned_pricer(sample)
print(f"Product: {sample.title[:60]}...")
print(f"Actual: ${sample.price:.2f}")
print(f"Fine-tuned prediction: {result}")

Product: Old Blood Noise Excess V2 Distortion Chorus/Delay Pedal...
Actual: $219.00
Fine-tuned prediction: $215.00


In [15]:
# Full evaluation on 200 test items
evaluate(fine_tuned_pricer, test, size=200)

  0%|          | 0/200 [00:00<?, ?it/s]

$30 $69 $29 $39 $20 $169 $95 $132 $11 $190 $410 $61 $4 $24 $7 $20 $91 $10 $202 $49 $11 $84 $1 $24 $7 $265 $255 $2 $10 $65 $65 $20 $30 $37 $35 $301 $26 $1 $31 $20 $177 $30 $2 $35 $45 $6 $4 $21 $55 $70 $20 $74 $269 $5 $47 $16 $4 $7 $6 $27 $160 $28 $52 $48 $454 $127 $94 $390 $33 $30 $21 $17 $120 $38 $25 $7 $135 $13 $9 $1 $45 $8 $19 $74 $2 $10 $56 $202 $55 $33 $8 $14 $4 $5 $6 $165 $13 $95 $205 $344 $10 $83 $12 $100 $73 $52 $25 $201 $0 $24 $43 $349 $24 $44 $34 $250 $1 $16 $44 $104 $6 $311 $7 $36 $75 $20 $22 $9 $19 $69 $51 $9 $24 $10 $166 $8 $35 $20 $62 $112 $12 $58 $30 $14 $87 $8 $2 $297 $48 $8 $13 $113 $29 $140 $111 $152 $55 $38 $72 $20 $90 $15 $16 $27 $95 $3 $211 $27 $10 $9 $3 $5 $100 $13 $53 $64 $7 $32 $27 $53 $258 $55 $237 $160 $6 $20 $79 $13 $27 $28 $18 $58 $45 $14 $10 $36 $0 $176 $27 $10 

## Step 5: Compare All Approaches

In [17]:
# After running all evaluations, manually record the MAE results here
# Update these values with actual results from the evaluations above

results = {
    "Constant (Mean)": 106.08,       # Update after running
    "Random Forest": 68.07,          # Update after running
    "XGBoost": 77.34,                # Update after running
    "GPT-4o-mini (zero-shot)": 47.89, # Update after running
    "GPT-4o-mini (fine-tuned)": 65.21,  # From Fine Tuned Pricer (MAE)
}

# Filter out None values (in case some haven't been run yet)
results = {k: v for k, v in results.items() if v is not None}

if results:
    fig = go.Figure(data=[go.Bar(
        x=list(results.keys()),
        y=list(results.values()),
        text=[f"${v:.2f}" for v in results.values()],
        textposition='outside',
        marker_color=['#636EFA', '#636EFA', '#636EFA', '#EF553B', '#00CC96'],
    )])
    fig.update_layout(
        title="Mean Absolute Error by Approach (lower is better)",
        yaxis_title="MAE ($)",
        width=900,
        height=500,
    )
    fig.show()
else:
    print("Run the evaluations above first, then fill in the results dict.")

## Summary

| Approach | MAE | Notes |
|----------|-----|-------|
| Constant (Mean) | 106.08 | Always predicts average price |
| Random Forest | 68.07 | 100 trees, CountVectorizer 2000 features |
| XGBoost | 77.34 | 500 trees, learning_rate=0.05 |
| GPT-4o-mini (zero-shot) | 47.89 | Via OpenRouter, no training |
| GPT-4o-mini (fine-tuned) | 65.21 | 400 train, 100 val, 1 epoch |

### Key Takeaways
- Traditional ML models provide a strong baseline without API costs
- Zero-shot LLMs can price products reasonably well using general knowledge
- Fine-tuning on domain data teaches the model price patterns specific to this dataset
- The evaluation framework (MAE + scatter plots) enables systematic comparison